# Intermediate 04 - Registration & Location

In this notebook:
- Learn how SIP REGISTER works
- Understand the role of `save()` and `lookup()`
- Learn about the Location table
- Practice registration expiry and renewal flows

---

## What is SIP Registration?

A SIP UA (phone, app) registers its current location (Contact URI) with a Registrar:

```
UA: REGISTER sip:example.com
    From: <sip:1001@example.com>
    Contact: <sip:1001@192.168.1.100:5060>
    Expires: 3600

Registrar: 200 OK
    (location table stores 1001 → 192.168.1.100)
```

When someone calls 1001, the Proxy looks up the location table
to find 1001's current address (`lookup`) and forwards the call.

## 1. Creating a REGISTER Message

In [ ]:
%%sip REGISTER
From: <sip:1001@example.com>;tag=reg01
To: <sip:1001@example.com>
Contact: <sip:1001@192.168.1.100:5060>

In [ ]:
# Check REGISTER details
$var(user) = $(fu{uri.user});
$var(contact) = $ct;
xlog("Registration request:");
xlog("  User: $var(user)");
xlog("  Contact: $var(contact)");

## 2. save() — Registering a Location

`save("location")` stores the Contact info from a REGISTER into the location table.

In real Kamailio it writes to an internal DB; here we simulate.

```kamailio
# Production code
if (!save("location")) {
    sl_reply_error();
}
```

In [ ]:
# save simulation
$var(user) = $(fu{uri.user});
xlog("Saving registration for $var(user)");
save("location");
xlog("Registration saved successfully");
send_reply(200, "OK");

## 3. lookup() — Finding a Location

When an INVITE arrives, `lookup("location")` finds the callee's current address.
On success, `$ru` (Request URI) is updated to the actual Contact URI.

```kamailio
# Production code
if (!lookup("location")) {
    sl_send_reply("404", "Not Found");
    exit;
}
# $ru now points to the registered Contact URI
```

In [ ]:
%%sip INVITE
From: <sip:2001@example.com>;tag=call1
To: <sip:1001@example.com>

In [ ]:
# lookup simulation
$var(callee) = $(ru{uri.user});
xlog("Looking up $var(callee) in location table");
lookup("location");

# Simulation: update $ru to the registered Contact
$ru = "sip:1001@192.168.1.100:5060";
xlog("Found! Routing to: $ru");

## 4. User Not Found — 404 Handling

Calling an unregistered user results in a 404 response.

In [ ]:
%%reset

In [ ]:
%%sip INVITE
From: <sip:2001@example.com>;tag=call2
To: <sip:9999@example.com>

In [ ]:
# lookup failure simulation
$var(callee) = $(ru{uri.user});
$var(registered) = "no";

xlog("Looking up $var(callee) in location table");

if ($var(registered) == "no") {
    xlog("User $var(callee) not found in location");
    send_reply(404, "Not Found");
} else {
    xlog("Found — routing to $ru");
}

## 5. Registration Expiry and Renewal

SIP registrations have an expiry time. The UA must re-REGISTER before expiry.

```
Register:   REGISTER (Expires: 3600)
            ↓ 1 hour passes
Expires:    Removed from location table
Renewal:    REGISTER (Expires: 3600) — re-register before expiry
```

Kamailio sets the default expiry with `modparam("usrloc", "default_expires", 3600)`.

In [ ]:
# Registration expiry simulation
$var(expiry) = 3600;
xlog("Registration expires in $var(expiry) seconds");

# Expiry check simulation
$var(remaining) = 100;
if ($var(remaining) > 0) {
    xlog("Registration still valid ($var(remaining) seconds remaining)");
} else {
    xlog("Registration EXPIRED — need to re-register");
    send_reply(423, "Interval Too Brief");
}

## 6. registered() — Check Registration Status

`registered("location")` quickly checks if a user is registered without a full DB lookup.

```kamailio
if (registered("location")) {
    # Already registered — update existing contact
} else {
    # New registration
}
```

In [ ]:
# registered check simulation
$var(user) = "1001";
$var(is_registered) = "yes";

xlog("Checking if $var(user) is registered");

if ($var(is_registered) == "yes") {
    xlog("$var(user) is already registered — updating contact");
} else {
    xlog("$var(user) is a new registration");
}

## 7. Full Register → Lookup → Call Flow

Complete flow of a production SIP proxy:

In [ ]:
# Step 1: 1001 registers
%%sip REGISTER
From: <sip:1001@example.com>;tag=r1
To: <sip:1001@example.com>
Contact: <sip:1001@10.0.0.5:5060>

In [ ]:
$var(user) = $(fu{uri.user});
xlog("[REGISTER] Saving location for $var(user)");
save("location");
send_reply(200, "OK");

In [ ]:
# Step 2: 2001 calls 1001
%%sip INVITE
From: <sip:2001@example.com>;tag=call01
To: <sip:1001@example.com>

In [ ]:
$var(callee) = $(ru{uri.user});
xlog("[INVITE] Looking up $var(callee)");
lookup("location");

# Simulation: update $ru from location lookup result
$ru = "sip:1001@10.0.0.5:5060";
xlog("[INVITE] Routing to: $ru");
record_route();
t_relay();

---

### Summary

| Concept | Description |
|---------|-------------|
| REGISTER | UA registers its current location with the server |
| save("location") | Store Contact info in the location table |
| lookup("location") | Find callee location, updates $ru on success |
| registered("location") | Quick check if user is registered |
| Expires | Registration expiry time (seconds) |
| 404 Not Found | User not registered |
| 423 Interval Too Brief | Expiry time too short |

Next notebook: **Advanced/01-dialog-and-failover.ipynb** →